# 01 – Exploration

Kurzer Blick auf die aufbereiteten Splits, bevor jemand am Training dreht. Trainiert wird nicht hier, sondern über `src/train.py`, `src/compare.py` und `src/finetune.py`.

Quellen, Lizenzen und die Split-Regel stehen in `data/README.md`. Eine Zeile ist ein Satz aus einer Finanznachricht, nicht ein ganzer Artikel. Die Dummy-Datei unter `data/sample/` ist nur für Tests und CI.


In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
os.environ["SENTIMENT_DATA"] = "prepared"

import pandas as pd
from src.config import LABEL_COLUMN, LANGUAGE_COLUMN, TEXT_COLUMN
from src.data import load_work_splits

train, val, test = load_work_splits()
train = train.copy(); train["split"] = "train"
val = val.copy(); val["split"] = "val"
test = test.copy(); test["split"] = "test"
df = pd.concat([train, val, test], ignore_index=True)
print(len(df), "Sätze", sorted(df["split"].unique()))
df.groupby("split").size()


8463 Sätze ['test', 'train', 'val']


split
test     1270
train    5923
val      1270
dtype: int64

## Klassen und Sprachen

Neutral ist in beiden Sprachen die größte Klasse. Der Split ist nach `label|language` stratifiziert, deshalb sieht jede Teilmenge ähnlich aus. Auswahlmaß bleibt macro-F1, damit die kleinere negative Klasse nicht untergeht.


In [2]:
print("Sprache")
display(df[LANGUAGE_COLUMN].value_counts())
print("Klasse")
display(df[LABEL_COLUMN].value_counts())
print("Klasse je Sprache")
display(pd.crosstab(df[LANGUAGE_COLUMN], df[LABEL_COLUMN]))
print("Klasse je Sprache und Split")
display(pd.crosstab([df["split"], df[LANGUAGE_COLUMN]], df[LABEL_COLUMN]))


Sprache


language
de    5023
en    3440
Name: count, dtype: int64

Klasse


label
neutral     4558
positive    2093
negative    1812
Name: count, dtype: int64

Klasse je Sprache


label,negative,neutral,positive
language,,,
de,1392,2425,1206
en,420,2133,887


Klasse je Sprache und Split


label           negative  neutral  positive
split language                             
test  de             209      364       181
      en              63      320       133
train de             974     1697       844
      en             294     1493       621
val   de             209      364       181
      en              63      320       133

## Textlänge

Zeichen, nicht Tokens. Die Sätze sind kurz. Ein Transformer mit Kontext 128 oder 256 sieht fast alles. Ganze Artikel in der API können darüber liegen: TF-IDF behält den vollen Text, die Transformer-Inferenz mittelt Chunks à höchstens 512 Tokens.


In [3]:
lengths = df[TEXT_COLUMN].str.len()
summary = lengths.groupby(df[LANGUAGE_COLUMN]).agg(
    n="count", min="min", median="median", mean="mean", p95=lambda s: s.quantile(0.95), max="max"
)
display(summary.round(1))
print("Sätze mit mehr als 1000 Zeichen:", int((lengths > 1000).sum()))


,n,min,median,mean,p95,max
language,,,,,,
de,5023,21,122.0,131.4,244.0,598
en,3440,20,116.0,125.1,231.0,315


Sätze mit mehr als 1000 Zeichen: 0


## Drei Beispiele

Nur zur Orientierung. Die Sammlung selbst bleibt lokal (`data/processed/`, nicht im Git). Phrasebank ist aus Investorsicht gelabelt, Kenpache gröber. Dieselbe Meldung kann als einzelne Sätze in Train und Test liegen, weil keine Artikel-ID mitgeliefert wird.


In [4]:
examples = (
    df.sort_values(TEXT_COLUMN)
    .groupby([LANGUAGE_COLUMN, LABEL_COLUMN], as_index=False)
    .head(1)
)
display(examples[[LANGUAGE_COLUMN, LABEL_COLUMN, "dataset", TEXT_COLUMN]])


,language,label,dataset,text
4974,de,negative,Kenpache/multilingual-financial-sentiment,"""""Eine wirklich richtig schlechte Entwicklung""..."
7900,de,positive,Kenpache/multilingual-financial-sentiment,"""""Miteinander reden"" beruhigt nervöse Asien-An..."
8157,de,neutral,Kenpache/multilingual-financial-sentiment,"""An der Frankfurter Börse waren die März-Zahle..."
274,en,neutral,takala/financial_phrasebank:sentences_75agree,( A spokesperson told WWD to expect a 50-50 mi...
1703,en,positive,takala/financial_phrasebank:sentences_75agree,"( ADP News ) - Dec 11 , 2008 - Finnish constru..."
6033,en,negative,takala/financial_phrasebank:sentences_75agree,"( ADP News ) - Feb 11 , 2009 - Finnish wood pr..."
